**Technical Challenge: The blind spots of Frontier Models in Hugging Face**

**SETTING UP THE COLAB ENVIRONMENT**

In [1]:
# REQUIREMENTS INSTALLATION
!nvidia-smi
!pip install -q transformers accelerate datasets huggingface_hub pandas

# UPGRADE TO THE LATEST TRANSFORMERS FROM GITHUB TO ALLOW Qwen3.5-4B
!pip install --upgrade transformers
!pip install git+https://github.com/huggingface/transformers.git

# LIBRARIES AND DEPENDENCIES IMPORT
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

Tue Mar  3 07:54:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**LOGGING INTO HUGGING FACE**

In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from huggingface_hub import whoami
SUCCESS = whoami()
print(f"{SUCCESS['name']}, you have successfully logged in.!")

olalytics, you have successfully logged in.!


**LOADING THE BASE MODEL (Qwen/Qwen3.5-4B) & INSTANTIATING THE TRANSFORMERS**

In [5]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

**FUNCTION TO GENERATE CONTEXT FROM THE MODEL**

In [6]:
def generate_context(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


**FOR THE PURPOSE OF THE CHALLENGE, TABULAR DATA WILL BE GENERATED AND SEND AS INPUT INTO THE MODEL TO GENERATE RESULT, THIS RESULT WILL BE CRITICIZE DEPENDING ON THE QUALITY OF THE OUTPUT**

In [7]:
one_shot_input = [
    {

        "model_input_question": """Product, Price, Quantity
        item1, 30, 102
        item2, 15, 110
        item3, 40, 90

        Which product has the highest total revenue?""",

        "accurate_result": "Product item3 (40*90=3600) is higher than Product item2 (15*110=1650) or Product item1 (30*102=3060)."
    }
  ]

In [8]:
results = []
for input in one_shot_input:
  response = generate_context(input["model_input_question"])
  print("INPUT:\n", input["model_input_question"])
  print("MODEL OUTPUT:\n", response)
  print("ACCURATE EXPECTED RESULT:\n", input["accurate_result"])
  print("="*100)

  results.append({
      "model_input_question": input["model_input_question"],
      "model_output": response,
      "accurate_result": input["accurate_result"]
  })

INPUT:
 Product, Price, Quantity
        item1, 30, 102
        item2, 15, 110
        item3, 40, 90
        
        Which product has the highest total revenue?
MODEL OUTPUT:
 Product, Price, Quantity
        item1, 30, 102
        item2, 15, 110
        item3, 40, 90
        
        Which product has the highest total revenue?
        A. item1
        B. item2
        C. item3
        D. Cannot be determined

<think>
Thinking Process:

1.  **Analyze the Request:** The user provides a table-like structure with three products (item1, item2, item3), each having a "Price" and a "Quantity". The goal is to determine which product has the highest total revenue.

2.  **Extract Data:**
    *   item1: Price = 30, Quantity = 102
    *   item2: Price = 15, Quantity = 
ACCURATE EXPECTED RESULT:
 Product item3 (40*90=3600) is higher than Product item2 (15*110=1650) or Product item1 (30*102=3060).


*With this simple single-shot, it is clearly that the blind spot in this model is that it failed to perform simple calculations on tabular data, failed to extract row entries from the tabular data and hence could not produce the accurate expected results.*

**CURATED TABULAR DATA FOR THE TEST USE CASES**

In [23]:
TABULAR_DATA = [
    {
        "model_input_question": "StudentName, StudentScore\nJerry, 80\nMark, 90\nOla, 70\n\nWhat is the average student score?",
        "accurate_result": "80."
    },
    {
        "model_input_question": "Country, GDP\nNigeria, 2.5\nGermany, 4.2\nItaly, 2.1\n\nWhich country has the highest GDP?",
        "accurate_result": "Germany."
    },
    {
        "model_input_question": "Date, Temperature\n2026-01-01, 1\n2026-01-02, 0\n2026-01-03, 4\n\nWhich day was the coldest?",
        "accurate_result": "2026-01-02."
    },
    {
        "model_input_question": "Name, Age, Smoker\nJohn, 34, Yes\nMary, 29, No\nSam, 41, Yes\n\nHow many smokers are in the table?",
        "accurate_result": "2 smokers."
    },
    {
        "model_input_question": "City, Population\nParis, 2148000\nBerlin, \nMadrid, 3223000\n\nWhich city has the smallest population?",
        "accurate_result": "Paris (Berlin is missing)."
    },
    {
        "model_input_question": "A | 10 | 3\nB | 5 | 8\nC | 12 | 2\n\nWhich row has the highest second column value?",
        "accurate_result": "Row C (12)."
    },
    {
        "model_input_question": "Item, Change\nA, 5\nB, -3\nC, 0\n\nWhich item has the lowest change?",
        "accurate_result": "B (-3)."
    },
    {
        "model_input_question": "ColumnX, ColumnY\n20, 5\n10, 15\n25, 45\n\nWhat is the sum of all values in ColumnY?",
        "accurate_result": "65."
    },
    {
        "model_input_question": "Product, Price\nItem1, 100\nItem2, 300\nItem3, 150\n\nWhat is the total of the 'Points' column?",
        "accurate_result": "There is no 'Points' column."
    },
    {
        "model_input_question": "Item, Value\nA, 10\nB, ten\nC, 20\n\nWhich item has the highest numeric value?",
        "accurate_result": "C (20)."
    },
    {
        "model_input_question": "Product, Cost, Tax\nItem1, 100, 20\nItem2, 50, 10\n\nWhat is the total cost including tax?",
        "accurate_result": "180 (120 + 60)."
    },
    {
        "model_input_question": "Product, Price, Discount\nitem1, 100, 10\nitem2, 80, 20\nitem3, 60, 5\n\nWhich product has the lowest final price after discount?",
        "accurate_result": "item3 (55)."
    },
    {
        "model_input_question": "Student, Score\nJane, 88\nMary, 98\nOla, 98\n\nWho scored highest?",
        "accurate_result": "Mary and Ola (tie)."
    },
    {
        "model_input_question": "Animal, Type\nDog, Mammal\nSnake, Reptile\nCat, Mammal\nFrog, Amphibian\n\nHow many mammals are listed?",
        "accurate_result": "2 mammals."
    },
    {
        "model_input_question": "Product, Price, Quantity\nitem1, 10, 3\nitem2, 5, 8\nitem3, 12, 2\n\nWhich product has the highest total revenue?",
        "accurate_result": "Product item2 (40)."
    },
]

**RUNNING THE MODEL ON ALL THE CURATED TABULAR DATA AND OBSERVING RESULTS**

In [24]:
MODEL_OUTPUT_AND_EXPECTED_RESULT = []

for input in TABULAR_DATA:
  response = generate_context(input["model_input_question"])
  print(f"Model Input Question:\n {input["model_input_question"]}")
  print(f"Model Response:\n {response}")
  print(f"Expected Result:\n {input["accurate_result"]}")
  print("+"*200)

  MODEL_OUTPUT_AND_EXPECTED_RESULT.append({
      "input": input["model_input_question"],
      "model_output": response,
      "expected_output": input["accurate_result"]
  })

Model Input Question:
 StudentName, StudentScore
Jerry, 80
Mark, 90
Ola, 70

What is the average student score?
Model Response:
 StudentName, StudentScore
Jerry, 80
Mark, 90
Ola, 70

What is the average student score?

<think>

</think>

To find the average student score, follow these steps:

1.  **Sum the scores**:
    $$80 + 90 + 70 = 240$$

2.  **Count the number of students**:
    There are 3 students (Jerry, Mark, and Ola).

3.  **Divide the total sum by the number of students**:
    $$240 \div 3 = 80$$

The average student score is **80**.
Expected Result:
 80.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Model Input Question:
 Country, GDP
Nigeria, 2.5
Germany, 4.2
Italy, 2.1

Which country has the highest GDP?
Model Response:
 Country, GDP
Nigeria, 2.5
Germany, 4.2
Italy, 2.1

Which country has the highest GDP?

<think>
Thin

**SAVING THE MODEL OUTPUT AND EXPECTED RESULT INTO THE JSON AND PARQUET USED FOR THE HUGGING FACE DATASET**

In [25]:
import json
import pandas as pd

In [26]:
with open("model_outputs.json", "w") as file:
  json.dump(MODEL_OUTPUT_AND_EXPECTED_RESULT, file, indent=2)

In [27]:
df = pd.DataFrame(MODEL_OUTPUT_AND_EXPECTED_RESULT)
df.to_parquet("model_outputs.parquet")

In [28]:
from datasets import Dataset
data = Dataset.from_list(MODEL_OUTPUT_AND_EXPECTED_RESULT)
data

Dataset({
    features: ['input', 'model_output', 'expected_output'],
    num_rows: 15
})

**CREATING DATASET REPOSITORY ON HUGGING FACE MANUALLY OR USING PYTHON DIRECTLY**

In [29]:
from huggingface_hub import create_repo

In [32]:
"""create_repo(
    repo_id="olalytics/qwen-tabular-data-blindspots",
    repo_type="dataset"
)"""

'create_repo(\n    repo_id="olalytics/qwen-tabular-data-blindspots",\n    repo_type="dataset"\n)'

**PUSHING DATASET TO HUGGING FACE**

In [31]:
# PUSHING THE DIRECTLY CREATED DATA WITHOUT SAVING TO FILE

HF_USERNAME = "olalytics"
DATASET_ID = f"{HF_USERNAME}/qwen-tabular-data-blindspots"
#data.push_to_hub(DATASET_ID)

**PUSHING DATASET TO HUGGING FACE USING THE STANDARD AND RECOMMENDED APPROACH (SAVING TO JSON OR PARQUET)**

In [30]:
from huggingface_hub import HfApi

In [ ]:
api = HfApi()

api.upload_file(
    path_or_fileobj="/content/model_outputs.json",
    path_in_repo="model_outputs.json",
    repo_id=DATASET_ID,
    repo_type="dataset"
)


CommitInfo(commit_url='https://huggingface.co/datasets/olalytics/qwen-tabular-data-blindspots/commit/c27beda16bd4ef1a1b60bf42b64482ac87a2d73e', commit_message='Upload model_outputs.json with huggingface_hub', commit_description='', oid='c27beda16bd4ef1a1b60bf42b64482ac87a2d73e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/olalytics/qwen-tabular-data-blindspots', endpoint='https://huggingface.co', repo_type='dataset', repo_id='olalytics/qwen-tabular-data-blindspots'), pr_revision=None, pr_num=None)

**END OF PROJECT - GO TO HUGGING FACE at:** *'https://huggingface.co/datasets/olalytics/qwen-tabular-data-blindspots/* **FOR DATASET README!.**